Create Summary CSVs
This notebook will create .csv files that contain summaries from multiple csvs of behavioral scoring data exported from ELAN. 
#Input: Put all .csv files (i.e. generated for each video and exported using ELAN) in a single directory. The format should be column 1: behavior name, column 2: blank, column 3: start time in sec.msec, column4: end time; column 5: duration. This script assumes 30 minute videos have been scored.
#The output will include a csv for cummulative time spent in a behavior over 5 minute bins in a 30 minute window for each unique behavior scored. It will also include a summary csv with the cumulative duration of each behavior for each video and 

Create seperate CSV with cumulative duration in 5 minute time bins over 30minute session for each behavior

In [11]:
#import relevant packages
import pandas as pd
import numpy as np
import os
import re

#Set Directory containing all CSV files
csv_path = r"/Volumes/Lab/Oxytocin Project/202603_OXTr_Antagonist_ACC/202601_OXTr_Antagonist_ACC_cohort2/data/SI/csv/Round1"
csv_path = os.path.abspath(csv_path)


Cumulative Duration:
Create a csv with cumulative time spent in each behavior over 5 minute bins over the 30 minute social interaction

In [12]:
# Defines a function to split behavior bouts that span across binned time intervals
def split_behavior_duration(start_time, duration, bins):
        split_durations = []
        
        end_time = start_time + duration
        for i in range(len(bins) - 1):
            bin_start = bins[i]
            bin_end = bins[i + 1]
            
            # Check if behavior overlaps with the current bin
            if start_time < bin_end and end_time > bin_start:
                # Find the overlap between the behavior and the current bin
                overlap_start = max(start_time, bin_start)
                overlap_end = min(end_time, bin_end)
                overlap_duration = overlap_end - overlap_start
                
                # Add the duration for the current bin if there is an overlap
                if overlap_duration > 0:
                    split_durations.append((i, overlap_duration))
        
        return split_durations

def createCumulativeDurationscsv(directory):
    summary_dir = os.path.join(csv_path, 'summary')
    os.makedirs(summary_dir, exist_ok=True)
    print("Summary directory:", summary_dir)

    cumulative_data = {}
    all_behaviors = set()

    # First pass: collect all unique behaviors across all files
    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            filepath = os.path.join(directory, filename)
            df = pd.read_csv(filepath, header=None)
            if not df.empty:
                behaviors = df[0].astype(str).str.lower().str.replace(r'[^a-z]', '', regex=True)
                all_behaviors.update(behaviors)

    # Second pass: process each file
    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            filepath = os.path.join(directory, filename)
            df = pd.read_csv(filepath, header=None)
            video_name = os.path.splitext(filename)[0]

            bins = np.arange(0, 1800 + 300, 300)
            bin_labels = [f'{int(b/60)}-{int(b/60 + 5)} min' for b in bins[:-1]]

            # Process file if not empty
            if not df.empty:
                df['behavior'] = df[0].astype(str).str.lower().str.replace(r'[^a-z]', '', regex=True)
                df['start_time'] = pd.to_numeric(df[2], errors='coerce')
                df['duration'] = pd.to_numeric(df[4], errors='coerce')

                cumulative_times = {behavior: np.zeros(len(bin_labels)) for behavior in df['behavior'].unique()}

                for index, row in df.iterrows():
                    behavior = row['behavior']
                    start_time = row['start_time']
                    duration = row['duration']
                    if pd.notnull(start_time) and pd.notnull(duration):
                        split_durations = split_behavior_duration(start_time, duration, bins)
                        for bin_index, bin_duration in split_durations:
                            cumulative_times[behavior][bin_index] += bin_duration

                cumulative_df = pd.DataFrame(cumulative_times, index=bin_labels).T
                cumulative_df = cumulative_df.cumsum(axis=1)
                cumulative_df.insert(0, 'video_name', video_name)
            else:
                # If file is empty, create an empty DataFrame
                cumulative_df = pd.DataFrame(columns=['video_name'] + bin_labels)

            # For every behavior in all_behaviors, ensure a row exists
            for behavior in all_behaviors:
                if behavior in cumulative_df.index:
                    behavior_df = cumulative_df.loc[[behavior]]
                else:
                    # Create a row of zeros for missing behavior
                    row = [video_name] + [0]*len(bin_labels)
                    behavior_df = pd.DataFrame([row], columns=['video_name'] + bin_labels)
                if behavior not in cumulative_data:
                    cumulative_data[behavior] = []
                cumulative_data[behavior].append(behavior_df)

    # Write one CSV per behavior, combining data across all videos
    for behavior, data_list in cumulative_data.items():
        behavior_df = pd.concat(data_list)
        behavior_df = behavior_df.sort_values(by='video_name')
        output_file = f'cumulative_{behavior}_times.csv'
        output_path = os.path.join(summary_dir, output_file)
        behavior_df.to_csv(output_path, index=False)
        print(f"Saved cumulative durations for {behavior} to {output_file}")

createCumulativeDurationscsv(csv_path)

Summary directory: /Volumes/Lab/Oxytocin Project/202603_OXTr_Antagonist_ACC/202601_OXTr_Antagonist_ACC_cohort2/data/SI/csv/Round1/summary
Saved cumulative durations for huddling to cumulative_huddling_times.csv
Saved cumulative durations for selfgrooming to cumulative_selfgrooming_times.csv
Saved cumulative durations for allolicking to cumulative_allolicking_times.csv
Saved cumulative durations for allogrooming to cumulative_allogrooming_times.csv


Total behavior duration:
create csv with total time spent doing each behavior for each video

In [3]:
def createDurationscsv(directory):
    # Create a 'summary' folder inside the input directory for the output CSV
    summary_dir = os.path.join(directory, 'summary')
    os.makedirs(summary_dir, exist_ok=True)

    behavior_durations_per_video = {}

    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            filepath = os.path.join(directory, filename)
            df = pd.read_csv(filepath, header=None)

            # Dictionary to hold total duration for each behavior in the current file
            behavior_total_duration = {}

            # Assuming the columns are: [Behavior, Unknown, Start Time, End Time, Duration]
            for _, row in df.iterrows():
                behavior = re.sub(r'[^a-z]', '', str(row[0]).strip().lower())
                duration = row[4]

                # Update the total duration for each behavior in the current video
                if behavior not in behavior_total_duration:
                    behavior_total_duration[behavior] = 0
                behavior_total_duration[behavior] += duration

            # Store the total durations for this video in the main dictionary
            # Use the filename (without extension) as the key
            behavior_durations_per_video[filename] = behavior_total_duration

    # Create a DataFrame to hold the summary of each video and behavior durations
    all_behaviors = set()
    for behavior_durations in behavior_durations_per_video.values():
        all_behaviors.update(behavior_durations.keys())
    
    all_behaviors = sorted(all_behaviors)  # Sort the behaviors for consistency

    # Create a list of dictionaries where each dictionary contains the video and its behavior durations
    summary_data = []
    for video, behavior_durations in behavior_durations_per_video.items():
        row_data = {'Video': video}
        for behavior in all_behaviors:
            row_data[behavior] = behavior_durations.get(behavior, 0)  # Fill with 0 if behavior not present
        summary_data.append(row_data)

    # Create the DataFrame from the summary data
    summary_df = pd.DataFrame(summary_data)
    summary_df=summary_df.sort_values(by='Video')
    # Save the summary DataFrame to a CSV file inside the 'summary' folder
    summary_csv = os.path.join(summary_dir, 'behavior_durations_summary.csv')
    summary_df.to_csv(summary_csv, index=False)

    print(f"Video behavior durations saved to {summary_csv}")

createDurationscsv(csv_path)


Video behavior durations saved to /Volumes/avn006/behaviorScoring/202409_FoodDeprivation/female/selfgroom/summary/behavior_durations_summary.csv


Latency to behavior:
Create a CSV containing the latency for each behavior
If behavior doesnt occur in a video, assumes 30 minute video and sets latency to 1800s (max)
i.e. take the first start time for each behavior

In [4]:
def createLatenciescsv(directory):
    summary_dir = os.path.join(directory, 'summary')
    os.makedirs(summary_dir, exist_ok=True)

    behavior_latencies_per_video = {}

    all_behaviors = set()

    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            filepath = os.path.join(directory, filename)
            df = pd.read_csv(filepath, header=None)

            behavior_latencies = {}

            for _, row in df.iterrows():
                behavior = re.sub(r'[^a-z]', '', str(row[0]).strip().lower())
                start_time = pd.to_numeric(row[2], errors='coerce')
                if pd.notnull(start_time):
                    if behavior not in behavior_latencies:
                        behavior_latencies[behavior] = start_time
                    else:
                        behavior_latencies[behavior] = min(behavior_latencies[behavior], start_time)
                all_behaviors.add(behavior)

            behavior_latencies_per_video[filename] = behavior_latencies

    # Fill missing behaviors with 1800
    for video, latencies in behavior_latencies_per_video.items():
        for behavior in all_behaviors:
            if behavior not in latencies:
                latencies[behavior] = 1800

    summary_path = os.path.join(summary_dir, 'behavior_latencies_summary.csv')
    summary_df = pd.DataFrame.from_dict(behavior_latencies_per_video, orient='index').sort_index()
    summary_df = summary_df[sorted(summary_df.columns)]  # Optional: sort columns alphabetically
    summary_df.to_csv(summary_path)

    print(f"Latency summary saved to: {summary_path}")

createLatenciescsv(csv_path)

Latency summary saved to: /Volumes/avn006/behaviorScoring/202409_FoodDeprivation/female/selfgroom/summary/behavior_latencies_summary.csv


Bouts of behavior
Create a CSV containing the total # of bouts or episodes of each behavior
Defines a bout as each episode of behavior at least 2 seconds apart (combines <2 seconds into one bout)

In [5]:
def createBoutscsv(directory):
    summary_dir = os.path.join(directory, 'summary')
    os.makedirs(summary_dir, exist_ok=True)

    behavior_bouts_per_video = {}
    all_behaviors = set()

    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            filepath = os.path.join(directory, filename)
            df = pd.read_csv(filepath, header=None)

             # Normalize behavior names and convert times
            df['behavior'] = df[0].astype(str).str.lower().str.replace(r'[^a-z]', '', regex=True)
            df['start_time'] = pd.to_numeric(df[2], errors='coerce')
            df['end_time'] = pd.to_numeric(df[3], errors='coerce')

            bout_counts = {}
            for behavior in df['behavior'].unique():
                behavior_df = df[df['behavior'] == behavior].sort_values(by='start_time')
                bout_count = 1 if not behavior_df.empty else 0  # Start with 1 if any bouts exist

                for i in range(len(behavior_df) - 1):
                    r1_end = behavior_df.iloc[i]['end_time']
                    r2_start = behavior_df.iloc[i + 1]['start_time']
                    if r2_start - r1_end > 2:
                        bout_count += 1
                bout_counts[behavior] = bout_count
                all_behaviors.add(behavior)

            behavior_bouts_per_video[filename] = bout_counts

    # Ensure every video has a value for every behavior (fill missing with 0)
    summary_data = []
    for video, bout_counts in behavior_bouts_per_video.items():
        row_data = {'Video': video}
        for behavior in sorted(all_behaviors):
            row_data[behavior] = bout_counts.get(behavior, 0)
        summary_data.append(row_data)

    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.sort_values(by='Video')
    summary_csv = os.path.join(summary_dir, 'behavior_bouts_summary.csv')
    summary_df.to_csv(summary_csv, index=False)
    print(f"Behavior bouts summary saved to {summary_csv}")

createBoutscsv(csv_path)

Behavior bouts summary saved to /Volumes/avn006/behaviorScoring/202409_FoodDeprivation/female/selfgroom/summary/behavior_bouts_summary.csv


rename files to indicate Round1 or Round2

In [4]:
import os
import re

# ===== USER VARIABLES =====
folder_path = r"/Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/202601_OXTr_Antagonist_ACC_cohort2/data/SI/csv/Round2"
cohort = "Cohort2"
default_round = "Round2"
# ==========================

for filename in os.listdir(folder_path):
    if not filename.endswith(".csv"):
        continue

    name, ext = os.path.splitext(filename)

    # Skip files already correctly formatted
    if re.match(r"Cohort\d+_Round\d+_", name):
        print(f"Skipped: {filename}")
        continue

    # Remove 'combined'
    name = name.replace("combined", "")

    # Look for Round# anywhere in the filename
    round_match = re.search(r"Round\d+", name)

    if round_match:
        round_tag = round_match.group()
        name = name.replace(round_tag, "")
    else:
        round_tag = default_round

    # Clean extra underscores
    name = re.sub(r"_+", "_", name).strip("_")

    # Build new filename
    new_filename = f"{cohort}_{round_tag}_{name}{ext}"

    old_path = os.path.join(folder_path, filename)
    new_path = os.path.join(folder_path, new_filename)

    os.rename(old_path, new_path)
    print(f"Renamed: {filename} -> {new_filename}")

Renamed: C1-R_RL_Round2_combined_KLT.csv -> Cohort2_Round2_C1-R_RL_KLT.csv
Renamed: C2-L_X_Round2_combined_KLT.csv -> Cohort2_Round2_C2-L_X_KLT.csv
Renamed: C2-R_RL_Round2_combined_KLT.csv -> Cohort2_Round2_C2-R_RL_KLT.csv
Renamed: C3-L_X_Round2_combined_KLT.csv -> Cohort2_Round2_C3-L_X_KLT.csv
Renamed: C3-R_RL_Round2_combined_KLT.csv -> Cohort2_Round2_C3-R_RL_KLT.csv
Renamed: C4-L_X_Round2_combined_KLT.csv -> Cohort2_Round2_C4-L_X_KLT.csv
Renamed: C4-R_RL_Round2_combined_KLT.csv -> Cohort2_Round2_C4-R_RL_KLT.csv
Renamed: C1-L_X_Round2_combined_KLT.csv -> Cohort2_Round2_C1-L_X_KLT.csv


change .csv so behaviors are strings and numbers are floats 

In [9]:
import os
import re

# ===== USER VARIABLES =====
folder_path = r"/Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts"
fixed_folder = os.path.join(folder_path, "fixed")
os.makedirs(fixed_folder, exist_ok=True)
# ==========================

for filename in os.listdir(folder_path):
    if not filename.endswith(".csv"):
        continue

    input_path = os.path.join(folder_path, filename)
    output_path = os.path.join(fixed_folder, filename)

    fixed_lines = []

    with open(input_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Split by tab or multiple spaces
            parts = re.split(r'\t+|\s{2,}', line)

            # Find first non-numeric part as behavior
            for i, part in enumerate(parts):
                part_clean = part.strip().replace('"', '')
                if not re.match(r'^-?\d+(\.\d+)?$', part_clean):
                    behavior = part_clean.replace('_', '-')
                    rest = parts[i+1:]
                    break
            else:
                # fallback
                behavior = parts[0]
                rest = parts[1:]

            # Convert numeric columns to float if possible
            numeric_rest = []
            for val in rest:
                val_clean = val.strip().replace('"','')
                try:
                    numeric_rest.append(str(float(val_clean)))
                except ValueError:
                    # keep as string if cannot convert
                    numeric_rest.append(val_clean)

            # Rebuild line: behavior + numeric columns separated by tabs
            fixed_line = '\t'.join([behavior] + numeric_rest)
            fixed_lines.append(fixed_line)

    # Save fixed CSV
    with open(output_path, "w") as out_f:
        for line in fixed_lines:
            out_f.write(line + "\n")

    print(f"Fixed CSV saved: {output_path}")

Fixed CSV saved: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/fixed/Cohort1_Round1_C1-L_X_postMEL_NK+KLT.csv
Fixed CSV saved: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/fixed/Cohort1_Round1_C1-R_RL_postMEL_PRB+KLT.csv
Fixed CSV saved: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/fixed/Cohort1_Round1_C2-L_X_postMEL_MW+KLT.csv
Fixed CSV saved: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/fixed/Cohort1_Round1_C2-R_RL_postMEL_NK+KLT.csv
Fixed CSV saved: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/fixed/Cohort1_Round1_C3-L_X_postMEL_MW+KLT.csv
Fixed CSV saved: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/fixed/Cohort1_Round1_C3-R_RL_postMEL_KLT+KLT.csv
Fixed CSV saved: /Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/fixed/Cohort1_Round1_C4-R_RL_postMEL_KLT+KLT.csv
Fixed CSV saved: /Volumes/L

Search all .csv files for any miss behaviors outside of allogrooming, allolicking, self-licking, selfgrooming, huddling --> broken rn 

In [15]:
# ...existing code...
import os
import csv

folder_path = r"/Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/fixed"

allowed_behaviors = {
    "allogrooming",
    "allolicking",
    "selfgrooming",
    "self-licking",
    "huddling"
}

unexpected_behaviors = set()

for filename in os.listdir(folder_path):
    if not filename.endswith(".csv"):
        continue

    file_path = os.path.join(folder_path, filename)
    file_invalid_behaviors = set()

    with open(file_path, newline='') as f:
        sample = f.read(2048)
        f.seek(0)
        try:
            dialect = csv.Sniffer().sniff(sample, delimiters=",\t")
        except csv.Error:
            dialect = csv.get_dialect('excel')
            dialect.delimiter = '\t' if '\t' in sample else ','
        reader = csv.reader(f, dialect)

        for row in reader:
            if not row:
                continue
            behavior = row[0]
            behavior_clean = behavior.strip().replace('"', '').replace('_', '-').lower()
            if behavior_clean and behavior_clean not in allowed_behaviors:
                file_invalid_behaviors.add(behavior_clean)
                unexpected_behaviors.add(behavior_clean)

    if file_invalid_behaviors:
        print(f"{filename} contains unexpected behaviors (rows may have issues): {file_invalid_behaviors}")

print("\nSummary of all unexpected behaviors across files:")
print(unexpected_behaviors)





Summary of all unexpected behaviors across files:
set()


Generate new 'filtered' folder that removes any instances of a behavior that are below a certain threshold duration 

In [3]:
import os
import re
import pandas as pd

# ===== USER VARIABLES =====
folder_path = r"/Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/fixed"
behavior_to_filter = "huddling"
duration_threshold = 3  # seconds
# ==========================

# robust reader that tries tab, comma, whitespace and falls back to manual split
def robust_read_csv(path):
    for sep, kwargs in [("\t", {}), (",", {}), (r"\s+", {"engine":"python"})]:
        try:
            df = pd.read_csv(path, sep=sep, header=None, **kwargs)
            if df.shape[1] > 1:
                return df, sep
        except Exception:
            pass
    # fallback: manual split by common delimiters
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    rows = [re.split(r'\t+|,|\s{2,}|\s+', line) for line in lines]
    df = pd.DataFrame(rows)
    return df, "\t"  # assume tab for fallback

# Create filtered output folder
filtered_folder = os.path.join(folder_path, "filtered")
os.makedirs(filtered_folder, exist_ok=True)

for filename in os.listdir(folder_path):
    if not filename.endswith(".csv"):
        continue

    input_path = os.path.join(folder_path, filename)
    output_path = os.path.join(filtered_folder, filename)

    df, sep = robust_read_csv(input_path)

    # determine duration column: prefer index 3 (column D), else pick column with most numeric values
    if df.shape[1] > 4:
        dur_col = 4
    else:
        numeric_counts = []
        for col in df.columns:
            numeric_counts.append((col, pd.to_numeric(df[col], errors='coerce').notna().sum()))
        dur_col = max(numeric_counts, key=lambda x: x[1])[0]

    # convert duration column to numeric and apply filter
    df[dur_col] = pd.to_numeric(df[dur_col], errors='coerce')
    filtered_df = df[~((df[0].astype(str).str.strip().str.lower() == behavior_to_filter) & (df[dur_col] < duration_threshold))]

    # Save filtered file (preserve original delimiter)
    filtered_df.to_csv(output_path, sep=sep, header=False, index=False)

    removed_rows = len(df) - len(filtered_df)
    print(f"{filename}: removed {removed_rows} rows")

print("\nFiltering complete. Files saved to:", filtered_folder)

Cohort1_Round1_C1-L_X_postMEL_NK+KLT.csv: removed 2 rows
Cohort1_Round1_C1-R_RL_postMEL_PRB+KLT.csv: removed 8 rows
Cohort1_Round1_C2-L_X_postMEL_MW+KLT.csv: removed 0 rows
Cohort1_Round1_C2-R_RL_postMEL_NK+KLT.csv: removed 2 rows
Cohort1_Round1_C3-L_X_postMEL_MW+KLT.csv: removed 1 rows
Cohort1_Round1_C3-R_RL_postMEL_KLT+KLT.csv: removed 2 rows
Cohort1_Round1_C4-R_RL_postMEL_KLT+KLT.csv: removed 0 rows
Cohort1_Round2_C1-L_X_postMEL_AVN+KLT.csv: removed 0 rows
Cohort1_Round2_C1-R_RL_postMEL_NK+KLT.csv: removed 0 rows
Cohort1_Round2_C2-L_X_postMEL_KPO+KLT.csv: removed 0 rows
Cohort1_Round2_C2-R_RL_postMEL_PRB+KLT+KLT.csv: removed 0 rows
Cohort1_Round2_C3-L_X_postMEL_MW+KLT.csv: removed 0 rows
Cohort1_Round2_C3-R_RL_postMEL_PRB+KLT.csv: removed 0 rows
Cohort1_Round2_C4-R_RL_postMEL_NK+KLT.csv: removed 0 rows
Cohort2_Round1_C1-L_X_NK.csv: removed 0 rows
Cohort2_Round1_C1-R_RL_NK.csv: removed 0 rows
Cohort2_Round1_C2-L_X_KLT.csv: removed 0 rows
Cohort2_Round1_C2-R_RL_KLT.csv: removed 0 rows

Create Raster Plot to represent behavior 

In [13]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams['font.family'] = 'Arial'

# ===== USER VARIABLES =====
# Use the filtered folder path from previous cell
plot_folder = r"/Volumes/Lab/Oxytocin/202603_OXTr_Antagonist_ACC/SI_analysis_both_cohorts/fixed/filtered"
session_duration = 1800  # seconds
# ==========================

# Define static colors for each behavior (consistent across all files)
static_behavior_colors = {
    'allogrooming': '#1f77b4',      # blue
    'allolicking': '#ff7f0e',       # orange
    'selfgrooming': '#2ca02c',      # green
    'self-licking': '#d62728',      # red
    'huddling': '#9467bd'           # purple
}

def get_color_for_behavior(behavior, known_colors, all_unique_behaviors):
    """Get color for a behavior, using predefined colors when available"""
    if behavior in known_colors:
        return known_colors[behavior]
    # For unknown behaviors, use tab20 palette
    unknown_behaviors = [b for b in sorted(all_unique_behaviors) if b not in known_colors]
    if unknown_behaviors:
        extra_colors = plt.cm.tab20(np.linspace(0, 1, len(unknown_behaviors)))
        idx = unknown_behaviors.index(behavior)
        return extra_colors[idx]
    return '#000000'  # black fallback

# Collect all unique behaviors first to ensure consistency
all_behaviors_set = set()
for filename in sorted(os.listdir(plot_folder)):
    if filename.endswith(".csv"):
        filepath = os.path.join(plot_folder, filename)
        df_temp, _ = robust_read_csv(filepath)
        if not df_temp.empty:
            all_behaviors_set.update(df_temp[0].astype(str).str.strip().unique())

# Create consistent color mapping for all behaviors
behavior_colors = {}
for behavior in all_behaviors_set:
    behavior_colors[behavior] = get_color_for_behavior(behavior, static_behavior_colors, all_behaviors_set)

# List to store processed data for combined plot
processed_data = []

# Process each CSV file in the filtered folder
for filename in sorted(os.listdir(plot_folder)):
    if not filename.endswith(".csv"):
        continue

    filepath = os.path.join(plot_folder, filename)
    df, sep = robust_read_csv(filepath)

    if df.empty:
        print(f"Skipping empty file: {filename}")
        continue

    # Extract behavior, start time, and end time columns
    df['behavior'] = df[0].astype(str).str.strip()
    df['start_time'] = pd.to_numeric(df[2], errors='coerce')
    df['end_time'] = pd.to_numeric(df[3], errors='coerce')

    # Remove rows with missing start or end times
    df = df.dropna(subset=['start_time', 'end_time'])

    if df.empty:
        print(f"No valid behavior data in: {filename}")
        continue

    # Get unique behaviors in this file
    unique_behaviors = sorted(df['behavior'].unique())

    # Store for combined plot
    title = os.path.splitext(filename)[0]
    title = re.sub(r'_postMEL.*', '', title)  # Remove _postMEL and anything after
    processed_data.append((title, df, unique_behaviors))

    # Create figure and axis
    fig, ax = plt.subplots(figsize=(16, 3))

    # Plot all behaviors on the same row with different colors (no edges)
    for _, row in df.iterrows():
        behavior = row['behavior']
        start = row['start_time']
        end = row['end_time']
        # Create a rectangle for each behavior occurrence (no edge, goes to axis edge)
        rect = mpatches.Rectangle((start, 0), end - start, 1,
                                 facecolor=behavior_colors[behavior],
                                 edgecolor='none')
        ax.add_patch(rect)

    # Set axis limits and labels
    ax.set_xlim(0, session_duration)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Time (minutes)', fontsize=12)

    # Set x-axis ticks in minutes (5-minute increments)
    xticks = np.arange(0, session_duration + 1, 300)  # 300 seconds = 5 minutes
    ax.set_xticks(xticks)
    ax.set_xticklabels([f'{int(x//60)}' for x in xticks])

    # Remove y-axis ticks and add black box around plot
    ax.set_yticks([])
    ax.spines['left'].set_visible(True)
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')
    ax.spines['right'].set_visible(True)
    ax.spines['right'].set_color('black')
    ax.spines['top'].set_visible(True)
    ax.spines['top'].set_color('black')

    # Add title (use filename without extension)
    ax.text(-0.1, 0.5, title, transform=ax.transAxes, fontsize=14, fontweight='bold', ha='right', va='center')

    # Create legend with only behaviors that appear in this file
    legend_patches = [mpatches.Patch(facecolor=behavior_colors[behavior], 
                                    label=behavior) for behavior in unique_behaviors]
    ax.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.01, 1))

    # Tight layout
    plt.tight_layout()

    # Save as PNG with the same name as the CSV file
    output_filename = os.path.splitext(filename)[0] + '.png'
    output_path = os.path.join(plot_folder, output_filename)
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved raster plot: {output_filename}")

    plt.close()

# Create combined plot with all individual plots stacked
if processed_data:
    nrows = len(processed_data)
    fig, axes = plt.subplots(nrows=nrows, figsize=(16, 3*nrows))

    if nrows == 1:
        axes = [axes]

    for i, (title, df, unique_behaviors) in enumerate(processed_data):
        ax = axes[i]

        # Plot rectangles
        for _, row in df.iterrows():
            behavior = row['behavior']
            start = row['start_time']
            end = row['end_time']
            rect = mpatches.Rectangle((start, 0), end - start, 1,
                                     facecolor=behavior_colors[behavior],
                                     edgecolor='none')
            ax.add_patch(rect)

        # Set limits
        ax.set_xlim(0, session_duration)
        ax.set_ylim(0, 1)

        # X-axis labels and ticks only on the bottom subplot
        if i == nrows - 1:
            ax.set_xlabel('Time (minutes)', fontsize=12)
            xticks = np.arange(0, session_duration + 1, 300)
            ax.set_xticks(xticks)
            ax.set_xticklabels([f'{int(x//60)}' for x in xticks])
        else:
            ax.set_xticks([])

        ax.set_yticks([])

        # Spines: top only on first, bottom only on last, left and right on all
        ax.spines['left'].set_visible(True)
        ax.spines['left'].set_color('black')
        ax.spines['bottom'].set_visible(i == nrows - 1)
        ax.spines['bottom'].set_color('black')
        ax.spines['right'].set_visible(True)
        ax.spines['right'].set_color('black')
        ax.spines['top'].set_visible(i == 0)
        ax.spines['top'].set_color('black')

        # Title
        ax.text(-0.1, 0.5, title, transform=ax.transAxes, fontsize=14, fontweight='bold', ha='right', va='center')

        # Legend
        legend_patches = [mpatches.Patch(facecolor=behavior_colors[behavior], label=behavior) for behavior in unique_behaviors]
        ax.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.01, 1))

    plt.tight_layout()
    combined_path = os.path.join(plot_folder, 'combined_raster_plots.png')
    plt.savefig(combined_path, dpi=300, bbox_inches='tight')
    print(f"Saved combined raster plot: {combined_path}")
    plt.close()

print("\nRaster plots complete. All PNG files saved to:", plot_folder)

Saved raster plot: Cohort1_Round1_C1-L_X_postMEL_NK+KLT.png
Saved raster plot: Cohort1_Round1_C1-R_RL_postMEL_PRB+KLT.png
Saved raster plot: Cohort1_Round1_C2-L_X_postMEL_MW+KLT.png
Saved raster plot: Cohort1_Round1_C2-R_RL_postMEL_NK+KLT.png
Saved raster plot: Cohort1_Round1_C3-L_X_postMEL_MW+KLT.png
Saved raster plot: Cohort1_Round1_C3-R_RL_postMEL_KLT+KLT.png
Saved raster plot: Cohort1_Round1_C4-R_RL_postMEL_KLT+KLT.png
Saved raster plot: Cohort1_Round2_C1-L_X_postMEL_AVN+KLT.png
Saved raster plot: Cohort1_Round2_C1-R_RL_postMEL_NK+KLT.png
Saved raster plot: Cohort1_Round2_C2-L_X_postMEL_KPO+KLT.png
Saved raster plot: Cohort1_Round2_C2-R_RL_postMEL_PRB+KLT+KLT.png
Saved raster plot: Cohort1_Round2_C3-L_X_postMEL_MW+KLT.png
Saved raster plot: Cohort1_Round2_C3-R_RL_postMEL_PRB+KLT.png
Saved raster plot: Cohort1_Round2_C4-R_RL_postMEL_NK+KLT.png
Saved raster plot: Cohort2_Round1_C1-L_X_NK.png
Saved raster plot: Cohort2_Round1_C1-R_RL_NK.png
Saved raster plot: Cohort2_Round1_C2-L_X_KLT